# Modul 6: Filter Spasial

Nama: Sahrul Ramadhani  
NIM: 244107020058  
Kelas: TI 3C

## D. Praktikum filter

### 1. Google Drive dan library

In [ ]:
from pathlib import Path
import importlib.util
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
from PIL import Image as im

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    from google.colab.patches import cv2_imshow
    drive.mount('/content/drive')
    IMAGE_DIR = Path('/content/drive/MyDrive/PCVK/Minggu 6/images')
else:
    IMAGE_DIR = Path.cwd() / 'images'

def read_image(filename):
    path = IMAGE_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f'File tidak ditemukan: {path}')
    image = cv.imread(str(path))
    if image is None:
        raise ValueError(f'Gambar tidak dapat dibaca: {path}')
    return image

### 2. Konvolusi tanpa library

In [ ]:
def convolution2d(image, kernel, stride=1, padding=0):
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError('Citra dan kernel harus berupa matriks dua dimensi')
    if not isinstance(stride, int) or stride < 1:
        raise ValueError('Stride harus bilangan bulat positif')
    if not isinstance(padding, int) or padding < 0:
        raise ValueError('Padding harus bilangan bulat tidak negatif')
    image_padded = np.pad(image, ((padding, padding), (padding, padding)), mode='constant')
    kernel_height, kernel_width = kernel.shape
    output_height = (image_padded.shape[0] - kernel_height) // stride + 1
    output_width = (image_padded.shape[1] - kernel_width) // stride + 1
    if output_height < 1 or output_width < 1:
        raise ValueError('Kernel lebih besar daripada citra setelah padding')
    output = np.zeros((output_height, output_width), dtype=np.float64)
    for y in range(output_height):
        for x in range(output_width):
            start_y = y * stride
            start_x = x * stride
            region = image_padded[start_y:start_y + kernel_height, start_x:start_x + kernel_width]
            output[y, x] = np.sum(region * kernel)
    return output

### 3. Grayscale dan sharpen

In [ ]:
img = read_image('mandrill.tiff')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
kernel_sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
hasil_sharpen = convolution2d(img_gray, kernel_sharpen, 1, 2)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Asli (Gray)')
plt.subplot(1, 2, 2)
plt.imshow(hasil_sharpen, cmap='gray', vmin=0, vmax=255)
plt.title('Sharpen')
plt.show()

### 4. Average, low pass, high pass, emboss, left Sobel, Canny, dan Gaussian 21x21

In [ ]:
kernel_average = np.ones((3, 3)) / 9
kernel_low_pass = np.array([[1, 1, 1], [1, 4, 1], [1, 1, 1]]) / 12
kernel_high_pass = np.array([[-1, 0, 1], [-1, 0, 3], [-3, 0, 1]])
kernel_emboss = np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]])
kernel_left_sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
kernel_canny = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])

kernel_size = 21
sigma = math.sqrt(kernel_size)
gaussian_kernel = cv.getGaussianKernel(kernel_size, sigma)
gauss_kernel = gaussian_kernel @ gaussian_kernel.transpose()

filter_kernels = [kernel_average, kernel_low_pass, kernel_high_pass, kernel_emboss, kernel_left_sobel, kernel_canny, gauss_kernel]
filter_titles = ['Average 3x3', 'Low Pass', 'High Pass', 'Emboss', 'Left Sobel Edge Detection', 'Canny Edge Detection', '21x21 Gaussian Blur']
manual_results = {}
for kernel, title in zip(filter_kernels, filter_titles):
    result = convolution2d(img_gray, kernel, 1, kernel.shape[0] // 2)
    manual_results[title] = result
    plt.figure(figsize=(6, 5))
    plt.imshow(result, cmap='gray', vmin=0, vmax=255)
    plt.title(title)
    plt.show()

## E. Filter library dan filter modern

### Percobaan 1: Gaussian, sharpen, dan Canny

In [ ]:
def show_side_by_side(images, titles, figsize=(15, 5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 2:
            plt.subplot(1, len(images), i + 1)
            plt.imshow(img, cmap='gray')
        else:
            plt.subplot(1, len(images), i + 1)
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis('off')
    plt.show()

img = read_image('lena.jpg')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
blur = cv.GaussianBlur(img, (7, 7), 1)
edges = cv.Canny(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 100, 200)
sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
sharpened = cv.filter2D(img, -1, sharpen_kernel)
show_side_by_side([img, blur, sharpened, edges], ['Asli', 'Gaussian Blur', 'Sharpened', 'Canny Edge Detection'])

### Percobaan 2: Bilateral dan edge preserving

In [ ]:
bilateral = cv.bilateralFilter(img, 50, 100, 100)
edge_preserve = cv.edgePreservingFilter(img, flags=1, sigma_s=100, sigma_r=0.9)
show_side_by_side([img, bilateral, edge_preserve], ['Asli', 'Bilateral Filter', 'Edge Preserving Filter'])

### Percobaan 3: Feature map CNN

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        return self.conv1(x)

for percobaan in range(3):
    model = SimpleCNN()
    img_tensor = torch.tensor(img_gray, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
    with torch.no_grad():
        features = model(img_tensor)
    feature_maps = [features[0, i].numpy() for i in range(features.shape[1])]
    print('Percobaan:', percobaan + 1)
    show_side_by_side([img_gray] + feature_maps, ['Asli (Gray)'] + [f'Feature {i + 1}' for i in range(len(feature_maps))])

Hasil feature map berubah pada setiap percobaan karena model baru memiliki bobot awal acak. Setiap kernel menonjolkan pola terang-gelap yang berbeda. Model ini belum dilatih, sehingga pola yang muncul belum menunjukkan hasil pengenalan objek.

### Percobaan 4: Beauty dan vintage

In [ ]:
smooth = cv.bilateralFilter(img, d=15, sigmaColor=75, sigmaSpace=75)
gaussian = cv.GaussianBlur(smooth, (0, 0), 3)
sharpened = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)
alpha = 1.2
beta = 15
beauty = cv.convertScaleAbs(sharpened, alpha=alpha, beta=beta)

sepia_kernel = np.array([[0.272, 0.534, 0.131], [0.349, 0.686, 0.168], [0.393, 0.769, 0.189]])
sepia = cv.transform(img, sepia_kernel)
sepia = np.clip(sepia, 0, 255).astype(np.uint8)
rows, cols = img.shape[:2]
kernel_x = cv.getGaussianKernel(cols, cols * 0.6)
kernel_y = cv.getGaussianKernel(rows, rows * 0.6)
kernel = kernel_y * kernel_x.T
mask = kernel / kernel.max()
vignette = np.copy(sepia)
for i in range(3):
    vignette[:, :, i] = vignette[:, :, i] * mask
noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)
show_side_by_side([img, beauty, old_img], ['Asli', 'Beauty Filter', 'Old/Vintage Filter'])

### Percobaan 5: Anime / cartoon

In [ ]:
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
gray_blur = cv.medianBlur(gray, 7)
edges = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 9, 9)
color = cv.bilateralFilter(img, d=9, sigmaColor=200, sigmaSpace=200)
cartoon = cv.bitwise_and(color, color, mask=edges)
show_side_by_side([img, cartoon], ['Asli', 'Cartoon Filter'])

### Percobaan 6: Night filter

In [ ]:
img = read_image('djawatan.jpg')
img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
night = cv.convertScaleAbs(img, alpha=0.6, beta=-40)
blue_tint = np.full_like(night, (50, 0, 100))
night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)
kernel = np.ones((15, 15), np.float32) / 225
glow = cv.filter2D(night, -1, kernel)
night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)
show_side_by_side([img, night, night_glow], ['Asli', 'Night Filter', 'Night + Glow (filter2D)'])

### Percobaan 7: Pagi dan kabut

In [ ]:
alpha = 0.9
beta = 20
soft = cv.convertScaleAbs(img, alpha=alpha, beta=beta)
warm_tint = np.full_like(soft, (40, 70, 120))
pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)
kernel = cv.getGaussianKernel(3, 3)
kernel = kernel @ kernel.T
kabut = cv.filter2D(pagi, -1, kernel)
white_layer = np.full_like(pagi, 255)
kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)
show_side_by_side([img, pagi, kabut], ['Asli', 'Pagi', 'Pagi + Kabut'])

## Tugas praktikum

### 1. Mean filter dan median filter pada salt and pepper

In [ ]:
img_tugas = read_image('lena.jpg')
gray_tugas = cv.cvtColor(img_tugas, cv.COLOR_BGR2GRAY)
rng = np.random.default_rng(6)
peluang = rng.random(gray_tugas.shape)
noisy = gray_tugas.copy()
noisy[peluang < 0.025] = 0
noisy[peluang > 0.975] = 255
mean_filtered = cv.blur(noisy, (3, 3))
median_filtered = cv.medianBlur(noisy, 3)
show_side_by_side([gray_tugas, noisy, mean_filtered, median_filtered], ['Asli (Gray)', 'Salt and Pepper', 'Mean 3x3', 'Median 3x3'])

Mean filter meratakan nilai piksel bersama bintik hitam-putih, sehingga bintik menyebar menjadi bercak abu-abu dan tepi ikut kabur. Median filter memilih nilai tengah dari sembilan piksel. Bintik yang terlalu gelap atau terang lebih mudah tersisih, sehingga gambar lebih bersih dan tepi lebih terjaga.

### 2. Sharpening 3x3 dan 5x5

In [ ]:
blurry = cv.GaussianBlur(gray_tugas, (7, 7), 1.5)
sharpen_3x3 = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=np.float32)
sharpen_5x5 = np.array([
    [0, 0, -1, 0, 0],
    [0, -1, -1, -1, 0],
    [-1, -1, 13, -1, -1],
    [0, -1, -1, -1, 0],
    [0, 0, -1, 0, 0]], dtype=np.float32)
sharp_3x3 = cv.filter2D(blurry, -1, sharpen_3x3)
sharp_5x5 = cv.filter2D(blurry, -1, sharpen_5x5)
show_side_by_side([blurry, sharp_3x3, sharp_5x5], ['Sedikit Buram', 'Sharpen 3x3', 'Sharpen 5x5'])

Kernel 3x3 memperjelas tepi dengan perubahan yang lebih ringan. Kernel 5x5 yang digunakan di sini memperkuat tepi lebih besar, tetapi juga lebih mudah membuat garis terang-gelap berlebihan dan memperkuat bintik kecil. Saya memilih kernel 3x3 untuk penggunaan sehari-hari karena hasilnya lebih wajar dan gangguan pada gambar lebih sedikit.

### 3. Fungsi kartun(image)

In [ ]:
def kartun(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)
    edges = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 9, 9)
    color = cv.bilateralFilter(image, d=9, sigmaColor=200, sigmaSpace=200)
    return cv.bitwise_and(color, color, mask=edges)

hasil_kartun = kartun(img_tugas)
show_side_by_side([img_tugas, hasil_kartun], ['Asli', 'Efek Kartun'])